# Retrieval v2 – Stage 5 ASR and Text-View Re-evaluation

Stage 5 is a selection stage. It does not recompute retrieval or fuse transcript channels. It evaluates the frozen Stage 4 winner independently on Whisper raw, Whisper processed, Parakeet raw, and Parakeet processed.


## 1. Setup


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount = False)
except Exception :
    pass

DEFAULT_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/"
    "aic26/asr_model_comparison"
)
ROOT_OVERRIDE = os.environ.get("AIC_RETRIEVAL_ROOT", "").strip()

ROOT = Path(ROOT_OVERRIDE).expanduser().resolve() if ROOT_OVERRIDE else DEFAULT_ROOT.resolve()
CODE_ROOT = ROOT
ARTIFACT_ROOT = ROOT

os.environ["AIC_RETRIEVAL_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)

RUNNER = CODE_ROOT / "src" / "07_evaluate_retrieval_v2.py"
CONFIG = CODE_ROOT / "configs" / "retrieval_v2.json"

print("Code root    :", CODE_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Runner exists:", RUNNER.exists())
print("Config exists:", CONFIG.exists())

assert RUNNER.exists()
assert CONFIG.exists()


Mounted at /content/drive
Code root    : /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Runner exists: True
Config exists: True


## 2. Stage 5 preflight


In [2]:
config = json.loads(CONFIG.read_text(encoding = "utf-8"))
decision = config["selection"]["stage04_decision"]
stage = config["stage05_selection"]

print("Stage 4 decision:", decision)
print("Quality thresholds:", json.dumps(stage["quality_thresholds"], indent = 2))
print("Cost thresholds   :", json.dumps(stage["cost_thresholds"], indent = 2))
print("Failure policy    :", json.dumps(stage["catastrophic_failure"], indent = 2))

assert decision["status"] == "frozen_after_review", (
    "Stage 5 is intentionally blocked. Freeze the Stage 4 winner in "
    "selection.stage04_decision first."
)
assert decision["selected_method"] in config["stage04_hybrid"]["methods"]

print("\nFrozen Stage 4 method:", decision["selected_method"])


Stage 4 decision: {'status': 'frozen_after_review', 'selected_method': 'H3_norm_25_75', 'retained_alternative': None, 'note': 'Selected after Stage 4 review. H3 improved composite Video MRR from 0.7301 for dense-only to 0.7841, improved Video R@1 from 0.6500 to 0.7125, improved Story MRR from 0.8413 to 0.8777, and had a positive 90% bootstrap interval for Video RR improvement versus dense-only.'}
Quality thresholds: {
  "video_recall_at_1_max_deficit": 0.05,
  "video_mrr_max_deficit": 0.03,
  "video_recall_at_5_max_deficit": 0.05,
  "video_recall_at_10_max_deficit": 0.05,
  "story_recall_at_1_max_deficit": 0.1,
  "story_mrr_max_deficit": 0.05
}
Cost thresholds   : {
  "minimum_rtf_speedup": 10.0
}
Failure policy    : {
  "minimum_rank_drop": 10,
  "result_rank_above": 10,
  "maximum_additional_failures": 1,
  "flag_reference_rank_1_candidate_above": 20
}

Frozen Stage 4 method: H3_norm_25_75


## 3. Execute Stage 5


In [3]:
STAGE = "stage05_selection"

command = [
    sys.executable,
    "-u",
    str(RUNNER),
    "--stage",
    STAGE,
]

print("Running:", " ".join(command), "\n")

process = subprocess.Popen(
    command,
    cwd = str(CODE_ROOT),
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,
    text = True,
    bufsize = 1,
)

assert process.stdout is not None

for line in process.stdout :
    print(line, end = "")

returncode = process.wait()

if (returncode != 0) :
    raise RuntimeError(f"Retrieval v2 failed with exit code {returncode}")


Running: /usr/bin/python3 -u /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/src/07_evaluate_retrieval_v2.py --stage stage05_selection 

AIC 2026 RETRIEVAL V2
Stage:         stage05_selection
Code root:     /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Reports:       /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/reports/retrieval_v2/stage05_selection

[1/6] Validating Stage 5 thresholds and frozen Stage 4 decision...
      done in 6.9s
[2/6] Validating development20, All50, and Stage 4 prerequisite...
      done in 4.5s
[3/6] Loading the frozen Stage 4 winner across four independent channels...
      done in 4.4s
[4/6] Checking channel completeness, thresholds, failures, and holdout exclusion...
      done in 0.0s
[

## 4. Load Stage 5 reports


In [4]:
REPORT_ROOT = ARTIFACT_ROOT / "reports" / "retrieval_v2" / "stage05_selection"

summary = json.loads((REPORT_ROOT / "stage_summary.json").read_text(encoding = "utf-8"))
validation = json.loads((REPORT_ROOT / "validation_summary.json").read_text(encoding = "utf-8"))
metrics = pd.read_csv(REPORT_ROOT / "retrieval_metrics.csv")
view_summary = pd.read_csv(REPORT_ROOT / "view_summary.csv")
view_queries = pd.read_csv(REPORT_ROOT / "view_query_comparison.csv")
pair_summary = pd.read_csv(REPORT_ROOT / "asr_pair_summary.csv")
pair_queries = pd.read_csv(REPORT_ROOT / "asr_pair_query_comparison.csv")
failures = pd.read_csv(REPORT_ROOT / "failure_cases.csv")
operational = pd.read_csv(REPORT_ROOT / "operational_comparison.csv")
latency_context = pd.read_csv(REPORT_ROOT / "retrieval_latency_context.csv")

print("Status                  :", "PASS" if summary["passed"] else "FAIL")
print("Frozen Stage 4 method   :", summary["selected_stage04_method"])
print("Min Parakeet RTF speedup:", summary["minimum_parakeet_rtf_speedup"])
print("Equivalence candidates  :", summary["parakeet_equivalence_candidates"])


Status                  : PASS
Frozen Stage 4 method   : H3_norm_25_75
Min Parakeet RTF speedup: 33.8306493091748
Equivalence candidates  : ['whisper_raw__vs__parakeet_raw', 'whisper_raw__vs__parakeet_processed', 'whisper_processed__vs__parakeet_raw', 'whisper_processed__vs__parakeet_processed']


## 5. Four-channel retrieval quality


In [5]:
display(
    metrics[
        [
            "model_id", "view",
            "video_recall_at_1", "video_recall_at_5", "video_recall_at_10", "video_mrr",
            "story_recall_at_1", "story_recall_at_5", "story_recall_at_10", "story_mrr",
        ]
    ].sort_values(["model_id", "view"], kind = "mergesort")
)


,model_id,view,video_recall_at_1,video_recall_at_5,video_recall_at_10,video_mrr,story_recall_at_1,story_recall_at_5,story_recall_at_10,story_mrr
0,parakeet_ctc_0_6b_vietnamese,processed,0.75,0.85,0.9,0.810185,0.80,0.95,1.00,0.858333
1,parakeet_ctc_0_6b_vietnamese,raw,0.70,0.85,0.9,0.780736,0.85,1.00,1.00,0.895000
2,whisper_large_v3,processed,0.70,0.85,0.9,0.773275,0.85,0.95,0.95,0.883712
3,whisper_large_v3,raw,0.70,0.80,0.9,0.772075,0.80,0.95,1.00,0.873810


## 6. Raw versus processed within each ASR


In [6]:
display(view_summary)

print("\nLargest raw/processed video-rank changes:")
display(
    view_queries.reindex(
        view_queries["video_rank_delta_processed_minus_raw"].abs()
        .sort_values(ascending = False)
        .index
    ).head(30)
)


,model_id,query_count,video_rr_delta_mean_processed_vs_raw,video_rr_delta_median_processed_vs_raw,video_better_processed,video_tie,video_worse_processed,story_rr_delta_mean_processed_vs_raw,story_better_processed,story_tie,story_worse_processed
0,parakeet_ctc_0_6b_vietnamese,20,0.029449,0.0,2,15,3,-0.036667,1,16,3
1,whisper_large_v3,20,0.001199,0.0,3,15,2,0.009903,2,15,3



Largest raw/processed video-rank changes:


,model_id,query_id,raw_story_rank,processed_story_rank,story_relation_processed_vs_raw,story_rr_delta_processed_vs_raw,raw_video_rank,processed_video_rank,video_relation_processed_vs_raw,video_rank_delta_processed_minus_raw,video_rr_delta_processed_vs_raw,top_story_changed,top_video_changed
16,parakeet_ctc_0_6b_vietnamese,R2-8,5,6,worse,-0.033333,11,27,worse,16,-0.053872,False,False
36,whisper_large_v3,R2-8,1,3,worse,-0.666667,32,25,better,-7,0.008750,True,True
30,whisper_large_v3,R2-1,1,1,tie,0.000000,13,17,worse,4,-0.018100,True,True
10,parakeet_ctc_0_6b_vietnamese,R2-1,1,2,worse,-0.500000,21,24,worse,3,-0.005952,True,False
2,parakeet_ctc_0_6b_vietnamese,R1-13,2,4,worse,-0.250000,2,1,better,-1,0.500000,False,True
18,parakeet_ctc_0_6b_vietnamese,R3-20,1,1,tie,0.000000,3,2,better,-1,0.166667,False,False
29,whisper_large_v3,R3-15,2,1,better,0.500000,1,2,worse,1,-0.500000,True,True
25,whisper_large_v3,R1-22,1,1,tie,0.000000,2,1,better,-1,0.500000,False,True
31,whisper_large_v3,R2-3,1,1,tie,0.000000,6,5,better,-1,0.033333,False,False
11,parakeet_ctc_0_6b_vietnamese,R2-3,1,1,tie,0.000000,7,8,worse,1,-0.017857,False,False


## 7. Whisper versus Parakeet threshold table


In [7]:
display(pair_summary)


,pair_id,whisper_view,parakeet_view,whisper_video_recall_at_1,parakeet_video_recall_at_1,video_recall_at_1_deficit,video_recall_at_1_passed,whisper_video_mrr,parakeet_video_mrr,video_mrr_deficit,...,story_mrr_deficit,story_mrr_passed,quality_thresholds_passed,catastrophic_regression_count,hard_top1_regression_count,catastrophic_gate_passed,manual_inspection_required,minimum_parakeet_rtf_speedup,cost_gate_passed,parakeet_equivalence_candidate
0,whisper_raw__vs__parakeet_raw,raw,raw,0.7,0.70,0.0,True,0.772075,0.780736,0.0,...,0.000000,True,True,0,0,True,False,33.830649,True,True
1,whisper_raw__vs__parakeet_processed,raw,processed,0.7,0.75,0.0,True,0.772075,0.810185,0.0,...,0.015476,True,True,1,0,True,False,33.830649,True,True
2,whisper_processed__vs__parakeet_raw,processed,raw,0.7,0.70,0.0,True,0.773275,0.780736,0.0,...,0.000000,True,True,0,0,True,False,33.830649,True,True
3,whisper_processed__vs__parakeet_processed,processed,processed,0.7,0.75,0.0,True,0.773275,0.810185,0.0,...,0.025379,True,True,0,0,True,False,33.830649,True,True


## 8. Operational cost, failure cases, and retrieval-latency context


In [8]:
print("ASR operational comparison:")
display(operational)

print("\nHigh-value Whisper/Parakeet failure and win cases:")
display(failures)

print("\nRetrieval latency context (separate measured scopes):")
display(latency_context)


ASR operational comparison:


,stage_id,whisper_aggregate_rtf,parakeet_aggregate_rtf,parakeet_rtf_speedup,whisper_total_inference_runtime_s,parakeet_total_inference_runtime_s,whisper_peak_gpu_memory_bytes,parakeet_peak_gpu_memory_bytes,whisper_to_parakeet_peak_memory_ratio,whisper_failed_windows,parakeet_failed_windows,whisper_empty_windows,parakeet_empty_windows
0,stage1_10_video,0.325698,0.008545,38.115390,4219.101901,110.692868,6.744244e+09,4.912931e+09,1.372754,0,0,0,0
1,stage1_extension40,0.304687,0.009006,33.830649,6775.823280,200.286528,6.745739e+09,4.921512e+09,1.370664,0,0,0,0



High-value Whisper/Parakeet failure and win cases:


,pair_id,whisper_view,parakeet_view,query_id,query_text,whisper_story_rank,parakeet_story_rank,story_rank_delta_parakeet_minus_whisper,whisper_story_rr,parakeet_story_rr,...,video_rank_delta_parakeet_minus_whisper,whisper_video_rr,parakeet_video_rr,video_rr_delta_parakeet_minus_whisper,whisper_video_margin,parakeet_video_margin,whisper_top_video_id,parakeet_top_video_id,catastrophic_regression,hard_top1_regression
0,whisper_processed__vs__parakeet_processed,processed,processed,R2-1,Nhiều người mặt áo cờ đỏ sao vàng đứng trước b...,1,2,1.0,1.000000,0.500000,...,7,0.058824,0.041667,-0.017157,-0.187100,-0.173930,K06_V014,K03_V023,False,False
1,whisper_processed__vs__parakeet_processed,processed,processed,R2-3,Đoạn clip về một gian trưng bày văn hóa - du l...,1,1,0.0,1.000000,1.000000,...,3,0.200000,0.125000,-0.075000,-0.050101,-0.127313,K05_V013,K13_V003,False,False
2,whisper_processed__vs__parakeet_processed,processed,processed,R2-8,Đoạn clip mở đầu với hình ảnh cây cầu có hệ th...,3,6,3.0,0.333333,0.166667,...,2,0.040000,0.037037,-0.002963,-0.155955,-0.212565,K01_V009,L21_V027,False,False
3,whisper_processed__vs__parakeet_processed,processed,processed,R1-2,Mẩu tin giới thiệu về đàn hổ tại một địa phươn...,1,1,0.0,1.000000,1.000000,...,0,1.000000,1.000000,0.000000,0.366365,0.286721,L21_V029,L21_V029,False,False
4,whisper_processed__vs__parakeet_processed,processed,processed,R1-9,Đoạn clip là cảnh thu hoạch dứa ở miền Tây: mộ...,11,4,-7.0,0.090909,0.250000,...,0,1.000000,1.000000,0.000000,0.004193,0.072993,L27_V013,L27_V013,False,False
5,whisper_processed__vs__parakeet_processed,processed,processed,R1-13,Đoạn video mô tả một người ngồi vệ sinh máy ản...,4,4,0.0,0.250000,0.250000,...,0,1.000000,1.000000,0.000000,0.052143,0.000614,L30_V095,L30_V095,False,False
6,whisper_processed__vs__parakeet_processed,processed,processed,R3-15,Trong một đoạn video nấu ăn một món ăn về tôm....,1,1,0.0,1.000000,1.000000,...,-1,0.500000,1.000000,0.500000,-0.006547,0.007774,L26_V178,L26_V222,False,False
7,whisper_processed__vs__parakeet_processed,processed,processed,R3-2,Cảnh quay cận cảnh một miếng thiếc có nhiều lỗ...,1,1,0.0,1.000000,1.000000,...,-4,0.166667,0.500000,0.333333,-0.096826,-0.016653,L30_V057,K01_V009,False,False
8,whisper_processed__vs__parakeet_raw,processed,raw,R2-1,Nhiều người mặt áo cờ đỏ sao vàng đứng trước b...,1,1,0.0,1.000000,1.000000,...,4,0.058824,0.047619,-0.011204,-0.187100,-0.137051,K06_V014,K03_V023,False,False
9,whisper_processed__vs__parakeet_raw,processed,raw,R2-3,Đoạn clip về một gian trưng bày văn hóa - du l...,1,1,0.0,1.000000,1.000000,...,2,0.200000,0.142857,-0.057143,-0.050101,-0.068301,K05_V013,K13_V003,False,False



Retrieval latency context (separate measured scopes):


,source,method_id,channel,metric,value_ms
0,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_processed,query_encode_p50_ms,24.436092
1,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_processed,query_encode_p90_ms,29.620544
2,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_processed,retrieval_p50_ms,0.386877
3,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_processed,retrieval_p90_ms,0.454827
4,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_processed,end_to_end_p50_ms,24.820841
5,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_processed,end_to_end_p90_ms,30.129896
6,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_raw,query_encode_p50_ms,24.436092
7,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_raw,query_encode_p90_ms,29.620544
8,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_raw,retrieval_p50_ms,0.364009
9,stage02_dense_online_reference,D1_e5_large_instruct,parakeet_raw,retrieval_p90_ms,0.463589


## 9. Final decision

The runner only identifies threshold/equivalence candidates. It does not choose an ASR or transcript view. Review the four channels, paired failures, and cost evidence, then freeze exactly one `selection.stage05_decision.selected_channel_id` together with its ASR and view.
